### Starting The End

In [1]:
import pandas as pd
import itertools
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
import joblib


In [2]:
import pandas as pd

def load_and_select(filepath: str, cols: list, id_col: str = "child_id") -> pd.DataFrame:
    """Read a CSV and keep only the needed columns, properly handling boolean and numeric fields."""
    
    # Read CSV
    df = pd.read_csv(filepath, dtype=str)  # read everything as string first
    
    # Strip whitespace
    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)


    
    # Keep only selected columns
    df = df[cols].copy()
    
    # Ensure child_id is clean
    df[id_col] = df[id_col].astype(str).str.strip()


    
    # Convert TRUE/FALSE strings to boolean
    for col in ["is_training_allowed", "isasd"]:
        if col in df.columns:
            df[col] = df[col].str.upper().map({"TRUE": True, "FALSE": False})

    
    # Convert numeric columns if they exist
    numeric_cols = [c for c in df.columns if "score" in c or c == "average_score"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Keep only rows allowed for training
    df = df[df["is_training_allowed"] == True]


    
    return df


In [3]:
game_columns = {
    "dance_doodle_game": [
        "age", "child_id", "cool_arms", "crossy_play", "happy_stand", "isasd", "is_training_allowed", "open_wings",
        "shh_fun", "silly_boxer", "stretch"
    ],
    "gaze_game": [
        "age", "child_id", "isasd", "is_training_allowed",
        "round1count", "round2count", "round3count"
    ],
    "gesture_game": [
        "age", "butterfly", "child_id", "closed_fist", "dua", "heart",
        "iloveyou", "isasd", "is_training_allowed", "open_palm", "pointing_up",
        "spectacle", "thumbs_down", "thumbs_up", "victory"
    ],
    "mirror_posture_game": [
        "age", "child_id", "isasd", "is_training_allowed", "kiss",
        "looking_sideways", "mouth_open", "showing_teeth"
    ],
    "repeat_with_me_game": [
        "age", "average_score", "child_id", "isasd",
        "is_training_allowed", "round1score", "round2score", "round3score",
        "round4score", "round5score", "round6score", "round7score",
        "round8score", "round9score", "round10score"
    ]
}


In [4]:
# Example filepaths - update with your actual CSV file paths
filepaths = {
    "dance_doodle_game": "C:/NeuroNurture/ALI_Model/dataset/dance_doodle_game.csv",
    "gaze_game": "C:/NeuroNurture/ALI_Model/dataset/gaze_game.csv",
    "gesture_game": "C:/NeuroNurture/ALI_Model/dataset/gesture_game.csv",
    "mirror_posture_game": "C:/NeuroNurture/ALI_Model/dataset/mirror_posture_game.csv",
    "repeat_with_me_game": "C:/NeuroNurture/ALI_Model/dataset/repeat_with_me_game.csv"
}

game_dfs = {
    game: load_and_select(path, game_columns[game])
    for game, path in filepaths.items()
}


In [19]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Example game list in lexicographic order
games = ["dance_doodle_game", "gaze_game", "gesture_game", "mirror_posture_game", "repeat_with_me_game"]

results = []

for i in range(1, 2**len(games)):
    # Convert i to 5-bit binary string
    bitmask = format(i, f"0{len(games)}b")[::-1]  # 00101

    # Determine which games to include based on bitmask
    subset = [games[j] for j in range(len(games)) if bitmask[j] == "1"]

    subset_name = "_".join(subset)
    #print(f"Training model on: {subset_name} (bitmask {bitmask})")

    # Start merging game data based on child_id

    merged = None
    for game in subset:
        df = game_dfs[game].copy()
        df = df[df["is_training_allowed"] == True]  # filter only allowed rows
        df = df.drop(columns=["session_id", "videourl", "date_time", "is_training_allowed"], errors="ignore")

        
        if merged is None or merged.empty:
            merged = df
        else:
            cols_to_merge = [c for c in df.columns if c not in merged.columns]
            merged = pd.merge(merged, df[["child_id"] + cols_to_merge], on="child_id", how="inner")

        
    if merged is None or merged.empty:
        continue


    
    
    X = merged.drop(columns=["child_id", "isasd"])
    y = merged["isasd"].astype(int)

    # Train-test split (e.g., 80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Model
    model = LogisticRegression(max_iter=1000)

    # Train the model
    model.fit(X_train, y_train)

    # Predict on test set
    y_pred = model.predict(X_test)

    # Compute accuracy
    acc = accuracy_score(y_test, y_pred)

    # Store the model and accuracy
    results.append({
        "subset": i,
        "bitmask": bitmask,
        "accuracy": acc,
        "model": model
    })
    print(f"Accuracy for {i}: {acc}")
import joblib
joblib.dump(results, "all_game_models.pkl")


Accuracy for 1: 0.5
Accuracy for 2: 1.0
Accuracy for 3: 1.0
Accuracy for 4: 1.0
Accuracy for 5: 0.5
Accuracy for 6: 0.5
Accuracy for 7: 0.5
Accuracy for 8: 0.5
Accuracy for 9: 0.5
Accuracy for 10: 1.0
Accuracy for 11: 0.5
Accuracy for 12: 0.5
Accuracy for 13: 0.5
Accuracy for 14: 0.5
Accuracy for 15: 0.5
Accuracy for 16: 1.0
Accuracy for 17: 1.0
Accuracy for 18: 1.0
Accuracy for 19: 1.0
Accuracy for 20: 1.0
Accuracy for 21: 0.5
Accuracy for 22: 0.5
Accuracy for 23: 1.0
Accuracy for 24: 1.0
Accuracy for 25: 1.0
Accuracy for 26: 1.0
Accuracy for 27: 1.0
Accuracy for 28: 0.5
Accuracy for 29: 1.0
Accuracy for 30: 1.0
Accuracy for 31: 1.0


['all_game_models.pkl']

In [8]:
print(results)

[{'subset': 1, 'bitmask': '10000', 'accuracy': 0.5, 'model': LogisticRegression(max_iter=1000)}, {'subset': 2, 'bitmask': '01000', 'accuracy': 1.0, 'model': LogisticRegression(max_iter=1000)}, {'subset': 3, 'bitmask': '11000', 'accuracy': 1.0, 'model': LogisticRegression(max_iter=1000)}, {'subset': 4, 'bitmask': '00100', 'accuracy': 1.0, 'model': LogisticRegression(max_iter=1000)}, {'subset': 5, 'bitmask': '10100', 'accuracy': 0.5, 'model': LogisticRegression(max_iter=1000)}, {'subset': 6, 'bitmask': '01100', 'accuracy': 0.5, 'model': LogisticRegression(max_iter=1000)}, {'subset': 7, 'bitmask': '11100', 'accuracy': 0.5, 'model': LogisticRegression(max_iter=1000)}, {'subset': 8, 'bitmask': '00010', 'accuracy': 0.5, 'model': LogisticRegression(max_iter=1000)}, {'subset': 9, 'bitmask': '10010', 'accuracy': 0.5, 'model': LogisticRegression(max_iter=1000)}, {'subset': 10, 'bitmask': '01010', 'accuracy': 1.0, 'model': LogisticRegression(max_iter=1000)}, {'subset': 11, 'bitmask': '11010', 'ac